In [1]:
import os
import pandas as pd

path_data = "C:\\Users\\yazid\\workbench-workspace\\data-wrangling\\capstone-dw-kelompok-11\\data\\raw\\bps\\2026-10-17\\sdg8_tpt.csv"
df_raw = pd.read_csv(path_data)

print("Jumlah baris awal:", len(df_raw))
print("Daftar kolom:", list(df_raw.columns))
display(df_raw.head(5))

Jumlah baris awal: 2000
Daftar kolom: ['var_id', 'wilayah_id', 'wilayah', 'turvar_id', 'turvar', 'tahun_id', 'tahun', 'periode_id', 'periode', 'nilai', 'satuan']


,var_id,wilayah_id,wilayah,turvar_id,turvar,tahun_id,tahun,periode_id,periode,nilai,satuan
0,543,1100,ACEH,0,Tidak ada,100,2000,191,Tahunan,4.80,Persen
1,543,1100,ACEH,0,Tidak ada,101,2001,191,Tahunan,7.71,Persen
2,543,1100,ACEH,0,Tidak ada,102,2002,191,Tahunan,9.34,Persen
3,543,1100,ACEH,0,Tidak ada,103,2003,191,Tahunan,8.97,Persen
4,543,1100,ACEH,0,Tidak ada,104,2004,191,Tahunan,9.35,Persen


In [3]:
# ==========================================
# PERBAIKAN: NORMALISASI HURUF KAPITAL & GRID TAHUN + PERIODE
# ==========================================
PROVINSI_38 = [
    "Aceh", "Sumatera Utara", "Sumatera Barat", "Riau", "Jambi", "Sumatera Selatan",
    "Bengkulu", "Lampung", "Kepulauan Bangka Belitung", "Kepulauan Riau",
    "DKI Jakarta", "Jawa Barat", "Jawa Tengah", "DI Yogyakarta", "Jawa Timur", "Banten",
    "Bali", "Nusa Tenggara Barat", "Nusa Tenggara Timur",
    "Kalimantan Barat", "Kalimantan Tengah", "Kalimantan Selatan", "Kalimantan Timur", "Kalimantan Utara",
    "Sulawesi Utara", "Sulawesi Tengah", "Sulawesi Selatan", "Sulawesi Tenggara", "Gorontalo", "Sulawesi Barat",
    "Maluku", "Maluku Utara", "Papua Barat", "Papua", "Papua Selatan", "Papua Tengah", "Papua Pegunungan", "Papua Barat Daya"
]

# Kamus khusus setelah di-.title() agar singkatan/kata hubung sesuai baku 38 provinsi
MAP_SETELAH_TITLE = {
    "Dki Jakarta": "DKI Jakarta",
    "Di Yogyakarta": "DI Yogyakarta",
    "D.I. Yogyakarta": "DI Yogyakarta",
    "Kep. Bangka Belitung": "Kepulauan Bangka Belitung",
    "Bangka Belitung": "Kepulauan Bangka Belitung",
    "Kep. Riau": "Kepulauan Riau",
}

df_work = df_raw.copy()

# 1. Pisahkan baris Indonesia
mask_indo = df_work[COL_WILAYAH].astype(str).str.strip().str.lower() == "indonesia"
df_indonesia = df_work[mask_indo].copy()
df_prov = df_work[~mask_indo].copy()

# 2. Normalkan huruf kapital -> Title Case lalu rapikan nama khusus
df_prov[COL_WILAYAH] = (
    df_prov[COL_WILAYAH]
    .astype(str)
    .str.strip()
    .str.title()
    .replace(MAP_SETELAH_TITLE)
)

belum_terpetakan = sorted(set(df_prov[COL_WILAYAH].dropna()) - set(PROVINSI_38))
print("Wilayah di luar 38 provinsi:", belum_terpetakan)

df_prov = df_prov[df_prov[COL_WILAYAH].isin(PROVINSI_38)].copy()

# 3. Rapikan tahun & periode (tanpa merata-ratakan Feb/Agu menjadi tahunan di Silver)
df_prov[COL_TAHUN] = df_prov[COL_TAHUN].astype(str).str.strip()
df_prov[COL_PERIODE] = df_prov[COL_PERIODE].astype(str).str.strip()

# Urutan logis periode (Februari dulu baru Agustus, atau fallback alfabetis)
urutan_periode_map = {"Februari": 1, "Agustus": 2}
periode_unik = sorted(
    df_prov[COL_PERIODE].dropna().unique(),
    key=lambda x: (urutan_periode_map.get(x, 99), x)
)
tahun_unik = sorted(df_prov[COL_TAHUN].dropna().unique())

# Buat kunci periode berurut tahun_periode untuk grid yang presisi
df_prov["tahun_periode"] = df_prov[COL_TAHUN] + "_" + df_prov[COL_PERIODE]
tahun_periode_unik = [
    f"{thn}_{prd}"
    for thn in tahun_unik
    for prd in periode_unik
    if f"{thn}_{prd}" in set(df_prov["tahun_periode"])
]

# Catat tahun/periode awal muncul tiap provinsi
awal_provinsi = (
    df_prov.groupby(COL_WILAYAH, as_index=False)[COL_TAHUN]
    .min()
    .rename(columns={COL_TAHUN: "tahun_mulai_ada"})
)

# 4. Bentuk grid 38 provinsi x kombinasi (tahun, periode) yang ada
kombinasi_tp = (
    df_prov[[COL_TAHUN, COL_PERIODE]]
    .drop_duplicates()
    .sort_values([COL_TAHUN, COL_PERIODE], key=lambda col: col.map(urutan_periode_map) if col.name == COL_PERIODE else col)
)

grid_prov = pd.DataFrame({COL_WILAYAH: PROVINSI_38})
grid_dasar = grid_prov.merge(kombinasi_tp, how="cross")

grid_lengkap = grid_dasar.merge(df_prov, on=[COL_WILAYAH, COL_TAHUN, COL_PERIODE], how="left")
grid_lengkap = grid_lengkap.merge(awal_provinsi, on=COL_WILAYAH, how="left")

# Tandai sel kosong: sebelum provinsi muncul vs missing setelah muncul
grid_lengkap["status_sel"] = "terisi"
mask_kosong = grid_lengkap[COL_NILAI].isna()
grid_lengkap.loc[mask_kosong, "status_sel"] = "kosong_atau_missing"

mask_belum_ada = mask_kosong & (grid_lengkap[COL_TAHUN] < grid_lengkap["tahun_mulai_ada"])
grid_lengkap.loc[mask_belum_ada, "status_sel"] = "belum_terbit_atau_sebelum_pemekaran"

# 5. Hitung persen sel terisi per tahun & metrik utama
rekap_per_tahun = (
    grid_lengkap.groupby(COL_TAHUN, dropna=False)[COL_NILAI]
    .agg(
        total_sel="size",
        sel_terisi=lambda s: int(s.notna().sum()),
        sel_kosong=lambda s: int(s.isna().sum()),
    )
    .reset_index()
    .sort_values(COL_TAHUN)
)
rekap_per_tahun["persen_terisi"] = (
    rekap_per_tahun["sel_terisi"] / rekap_per_tahun["total_sel"] * 100
).round(2)

baris_silver = int(len(df_prov))
sel_kosong_grid = int(grid_lengkap[COL_NILAI].isna().sum())

print("Jumlah baris Indonesia dipisah:", len(df_indonesia))
print("baris_silver:", baris_silver)
print("sel_kosong_grid:", sel_kosong_grid)
print("Sel belum muncul / sebelum pemekaran:", int(mask_belum_ada.sum()))
display(awal_provinsi)
display(rekap_per_tahun)

Wilayah di luar 38 provinsi: []
Jumlah baris Indonesia dipisah: 61
baris_silver: 1939
sel_kosong_grid: 379
Sel belum muncul / sebelum pemekaran: 378


,wilayah,tahun_mulai_ada
0,Aceh,1986
1,Bali,1986
2,Banten,2001
3,Bengkulu,1986
4,DI Yogyakarta,1986
5,DKI Jakarta,1986
6,Gorontalo,2001
7,Jambi,1986
8,Jawa Barat,1986
9,Jawa Tengah,1986


,tahun,total_sel,sel_terisi,sel_kosong,persen_terisi
0,1986,38,26,12,68.42
1,1987,38,26,12,68.42
2,1988,38,26,12,68.42
3,1989,38,26,12,68.42
4,1990,38,26,12,68.42
5,1991,38,26,12,68.42
6,1992,38,26,12,68.42
7,1993,38,26,12,68.42
8,1994,38,26,12,68.42
9,1996,38,26,12,68.42


In [5]:
os.makedirs("C:\\Users\\yazid\\workbench-workspace\\data-wrangling\\capstone-dw-kelompok-11\\reports\\profil", exist_ok=True)
rekap_path = "C:\\Users\\yazid\\workbench-workspace\\data-wrangling\\capstone-dw-kelompok-11\\reports\\profil\\kelengkapan_panel_sdg8_tpt.csv"
rekap_per_tahun.to_csv(rekap_path, index=False)
print("Rekap kelengkapan panel tersimpan di:", rekap_path)

log_path = "../log-keputusan.csv"

keputusan_tahap3 = [
    {
        "id": "K-002",
        "tanggal": "2026-11-19",
        "nim": "124450081",
        "tahap": "Tahap 3",
        "seri_kode": "sdg8_tpt",
        "keputusan": "Pisahkan baris Indonesia dari grid 38 provinsi dan pertahankan periode asli (Februari/Agustus) tanpa merata-ratakan ke tahunan di Silver",
        "alternatif_ditolak": "Menggabungkan baris Indonesia ke grid 38 provinsi atau merata-ratakan Februari & Agustus menjadi tahunan di lapisan Silver",
        "alasan": "Menghindari pencampuran level nasional dengan provinsi dan menjaga granularitas observasi asli di Silver",
        "dampak_baris": baris_silver,
        "baris_terdampak": f"{len(df_indonesia)} baris Indonesia dipisah; {baris_silver} baris provinsi di grid",
    },
    {
        "id": "K-003",
        "tanggal": "2026-11-19",
        "nim": "124450081",
        "tahap": "Tahap 3",
        "seri_kode": "sdg8_tpt",
        "keputusan": "Bedakan sel kosong akibat seri belum terbit / provinsi belum terbentuk (pemekaran) dari missing value aktual",
        "alternatif_ditolak": "Menganggap seluruh sel kosong pada grid 38 provinsi x periode sebagai missing value seragam",
        "alasan": "Sel sebelum provinsi terbentuk (pemekaran) atau sebelum seri muncul bukan missing value operasional",
        "dampak_baris": sel_kosong_grid,
        "baris_terdampak": f"{int(mask_belum_ada.sum())} sel sebelum muncul dari total {sel_kosong_grid} sel kosong grid",
    },
]

if os.path.exists(log_path):
    df_log = pd.read_csv(log_path, dtype={"nim": str})
    df_log["nim"] = df_log["nim"].astype(str).str.replace(r"\.0$", "", regex=True)
    # Hapus baris lama Tahap 3 yang kosong id-nya jika sempat masuk, lalu append yang baru
    df_log = df_log[~(df_log["tahap"].eq("Tahap 3") & df_log["id"].isna())].copy()
    df_log = pd.concat([df_log, pd.DataFrame(keputusan_tahap3)], ignore_index=True)
else:
    df_log = pd.DataFrame(keputusan_tahap3)

df_log.to_csv(log_path, index=False)
display(pd.read_csv(log_path, dtype={"nim": str}).tail(3))

Rekap kelengkapan panel tersimpan di: C:\Users\yazid\workbench-workspace\data-wrangling\capstone-dw-kelompok-11\reports\profil\kelengkapan_panel_sdg8_tpt.csv


,id,tanggal,nim,tahap,seri_kode,keputusan,alternatif_ditolak,alasan,dampak_baris,baris_terdampak
0,K-001,2026-11-08,124450081,ekstraksi,semua,Wilayah diseragamkan ke susunan 38 provinsi ta...,Menggabungkan provinsi baru ke provinsi induk ...,Mengikuti aturan baku susunan 38 provinsi BPS ...,0.0,NaN
1,K-002,2026-11-19,124450081,Tahap 3,sdg8_tpt,Pisahkan baris Indonesia dari grid 38 provinsi...,Menggabungkan baris Indonesia ke grid 38 provi...,Menghindari pencampuran level nasional dengan ...,1939.0,61 baris Indonesia dipisah; 1939 baris provins...
2,K-003,2026-11-19,124450081,Tahap 3,sdg8_tpt,Bedakan sel kosong akibat seri belum terbit / ...,Menganggap seluruh sel kosong pada grid 38 pro...,Sel sebelum provinsi terbentuk (pemekaran) ata...,379.0,378 sel sebelum muncul dari total 379 sel koso...
